In [3]:
!uv pip install --system graphviz

Using Python 3.11.0 environment at: /home/michael/anaconda3/envs/tauso
Resolved 1 package in 255ms                                          
Installed 1 package in 29ms                                 
 + graphviz==0.21


In [10]:
import graphviz

# Create digraph with a Top-to-Bottom layout
dot = graphviz.Digraph('TAUSO_Workflow_Rect', comment='TAUSO Machine Learning Pipeline')

# Global attributes for a tight, rectangular look
dot.attr(rankdir='TB', splines='ortho', nodesep='0.3', ranksep='0.5')
dot.attr('node', shape='box', style='filled, rounded', fontname='Arial, Helvetica, sans-serif', fontsize='11', margin='0.15,0.1')
dot.attr('edge', fontname='Arial, Helvetica, sans-serif', fontsize='10', color='#555555', penwidth='1.5')

# ==========================================
# PHASE 1: Data Preparation & Engineering
# ==========================================
with dot.subgraph(name='cluster_data') as c:
    c.attr(label='Data Preparation & Engineering', style='rounded, dashed', color='#888888', fontname='Arial-Bold', fontsize='12', bgcolor='#FAFAFA', margin='15')

    c.node('data', label='<<table border="0" cellborder="0" cellpadding="1"><tr><td align="center"><b>Data Acquisition: OligoAtlas</b></td></tr><tr><td align="center">~170,000 ASO sequences<br/>(Empirical inhibition scores)</td></tr></table>>',
           fillcolor='#D6EAF8', shape='cylinder')

    c.node('prep', label='<<table border="0" cellborder="0" cellpadding="1"><tr><td align="center"><b>Rigorous Filtering</b></td></tr><tr><td align="left">- Standardizing genes (GRCh38)<br align="left"/>- Retaining 2\'-MOE &amp; cEt gapmers<br align="left"/>- Deduplication &amp; outlier exclusion</td></tr></table>>',
           fillcolor='#D6EAF8')

    c.node('feat', label='<<table border="0" cellborder="0" cellpadding="1"><tr><td align="center"><b>Feature Engineering (5 Classes)</b></td></tr><tr><td align="left">- Thermodynamics &amp; RNA-folding<br align="left"/>- Target accessibility<br align="left"/>- Specificity (Off-target)<br align="left"/>- Sequence-level motifs<br align="left"/>- Genomic context</td></tr></table>>',
           fillcolor='#E8DAEF')

    # Grid layout for Phase 1
    with c.subgraph() as s:
        s.attr(rank='same')
        s.node('data')
        s.node('prep')

    c.edge('data', 'prep')
    # Use an invisible edge to keep 'feat' centered below the first row
    c.edge('data', 'feat', style='invis')
    c.edge('prep', 'feat')

# ==========================================
# PHASE 2: Model Development
# ==========================================
with dot.subgraph(name='cluster_model') as c:
    c.attr(label='Model Development', style='rounded, dashed', color='#888888', fontname='Arial-Bold', fontsize='12', bgcolor='#FAFAFA', margin='15')

    c.node('tune', label='<<table border="0" cellborder="0" cellpadding="1"><tr><td align="center"><b>Hyperparameter Optimization</b></td></tr><tr><td align="center">Optuna framework<br/>(Maximizing Spearman correlation)</td></tr></table>>',
           fillcolor='#FCF3CF')

    c.node('sel', label='<<table border="0" cellborder="0" cellpadding="1"><tr><td align="center"><b>Feature Selection</b></td></tr><tr><td align="center">XGBoost-driven RFE<br/>(Dynamic pruning via info gain)</td></tr></table>>',
           fillcolor='#FCF3CF')

    c.node('train', label='<<table border="0" cellborder="0" cellpadding="1"><tr><td align="center"><b>Model Training</b></td></tr><tr><td align="center">XGBoost Regression Model<br/>(Excluding MD simulations)</td></tr></table>>',
           fillcolor='#FDEBD0')

    c.node('eval', label='<<table border="0" cellborder="0" cellpadding="1"><tr><td align="center"><b>Model Evaluation</b></td></tr><tr><td align="center">Assessed via Spearman correlation,<br/>MAE, &amp; RMSE across cohorts</td></tr></table>>',
           fillcolor='#FDEBD0')

    with c.subgraph() as s1:
        s1.attr(rank='same')
        s1.node('tune')
        s1.node('sel')

    with c.subgraph() as s2:
        s2.attr(rank='same')
        s2.node('train')
        s2.node('eval')

    c.edge('tune', 'sel')
    c.edge('sel', 'train')
    c.edge('train', 'eval')
    # Invisible edge for structural integrity
    c.edge('tune', 'train', style='invis')

# ==========================================
# PHASE 3: Application & Validation
# ==========================================
with dot.subgraph(name='cluster_app') as c:
    c.attr(label='Application & Validation', style='rounded, dashed', color='#888888', fontname='Arial-Bold', fontsize='12', bgcolor='#FAFAFA', margin='15')

    c.node('rank', label='<<table border="0" cellborder="0" cellpadding="1"><tr><td align="center"><b>Candidate Prediction</b></td></tr><tr><td align="center">Sliding-window screening<br/>and efficacy ranking</td></tr></table>>',
           fillcolor='#D4EFDF')

    c.node('topk', label='<<table border="0" cellborder="0" cellpadding="1"><tr><td align="center"><b>Top K Candidates</b></td></tr><tr><td align="center">Extraction of highest-ranked<br/>ASO sequences</td></tr></table>>',
           fillcolor='#D4EFDF', shape='parallelogram')

    c.node('val', label='<<table border="0" cellborder="0" cellpadding="1"><tr><td align="center"><b>Experimental Validation</b></td></tr><tr><td align="center">In vitro testing on MALAT1<br/>&amp; endogenous targets</td></tr></table>>',
           fillcolor='#A9DFBF')

    with c.subgraph() as s:
        s.attr(rank='same')
        s.node('rank')
        s.node('topk')

    c.edge('rank', 'topk')
    c.edge('topk', 'val')
    # Invisible edge for structural integrity
    c.edge('rank', 'val', style='invis')

# ==========================================
# Connect the Phases structurally
# ==========================================
# We connect the left-most nodes of each cluster to prevent diagonal shifting
dot.edge('feat', 'tune', weight='10')
dot.edge('train', 'rank', weight='10')

# Render the graph
dot.render('tauso_workflow_rect', format='png', cleanup=False)

'tauso_workflow_rect.png'